In [5]:
# 🔹 1. RELOAD AUTOMÁTICO
# POR QUÉ: Jupyter cachea imports. Si modificas loader.py y no reinicias el kernel,
# sigue usando la versión vieja. autoreload 2 evita reinicios manuales y garantiza
# que ejecutas siempre el código actualizado.
%load_ext autoreload
%autoreload 2

# 🔹 2. RESOLUCIÓN DE RUTA RAÍZ
# POR QUÉ: El notebook corre en notebooks/. Python resuelve rutas desde ahí.
# Esta línea busca la carpeta config/ en el CWD actual; si no está, sube un nivel.
# Garantiza que PROJECT_ROOT siempre apunte a Concurso_Data/, sin importar dónde abras el .ipynb.
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

# 🔹 3. INYECCIÓN DE sys.path
# POR QUÉ: Python no busca automáticamente en src/. Sin esto, from loader import DataLoader
# falla con ModuleNotFoundError cuando el CWD es notebooks/. Se inyecta solo para esta sesión.
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# 🔹 4. INSTANCIA DEL LOADER
# POR QUÉ: Pasamos la ruta absoluta del JSON para que el DataLoader no adivine rutas relativas.
from loader import DataLoader
CONFIG_PATH = str(PROJECT_ROOT / "config" / "mapping_config.json")
loader = DataLoader(config_path=CONFIG_PATH)

# 🔹 5. CARGA DE LOS 4 DATASETS
# POR QUÉ: Tu Etapa 1 exige cargar los 4 datasets. DS4 debe validarse estructuralmente ahora
# (columnas clave, existencia de archivo). Si falla, el pipeline se detiene antes de consumir RAM innecesaria.
raw = loader.load_all_datasets(['violencia', 'sexuales', 'poblacion', 'forense', 'seforense'])

# 🔹 6. VERIFICACIÓN DE SALIDA
# POR QUÉ: Confirma que la Etapa 1 cerró correctamente: 4 DataFrames en RAM, dimensiones esperadas,
# columnas clave presentes. Si alguna falla, el log del loader ya indicó el motivo exacto.
print("\n📊 RESULTADOS ETAPA 1:")
for k, v in raw.items():
    print(f"{k.capitalize():<12}: {v.shape} | Primeras cols: {v.columns[:15].tolist()}")

2026-06-10 10:17:52,283 - INFO - 📁 DataLoader inicializado en: /Users/anaaguirre/Documents/Cicatrices_invisibles/data
2026-06-10 10:17:52,284 - INFO - 🔍 Anclado a raíz de proyecto: /Users/anaaguirre/Documents/Cicatrices_invisibles
2026-06-10 10:17:52,284 - INFO - 🚀 Cargando 5 dataset(s)...


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2026-06-10 10:17:52,654 - INFO - ✅ violencia cargado: 682,558 filas, 8 cols
2026-06-10 10:17:52,956 - INFO - ✅ sexuales cargado: 392,576 filas, 9 cols
2026-06-10 10:19:10,665 - INFO - ✅ poblacion cargado: 84,233 filas, 312 cols
2026-06-10 10:19:12,062 - INFO - ✅ forense cargado: 236,840 filas, 36 cols
2026-06-10 10:19:12,897 - INFO - ✅ seforense cargado: 233,352 filas, 32 cols
2026-06-10 10:19:12,897 - INFO - ✅ Carga completada: 5/5 exitosos.



📊 RESULTADOS ETAPA 1:
Violencia   : (682558, 8) | Primeras cols: ['DEPARTAMENTO', 'MUNICIPIO', 'CODIGO DANE', 'ARMAS MEDIOS', 'FECHA HECHO', 'GENERO', 'GRUPO ETARIO', 'CANTIDAD']
Sexuales    : (392576, 9) | Primeras cols: ['DEPARTAMENTO', 'MUNICIPIO', 'CODIGO DANE', 'ARMAS MEDIOS', 'FECHA HECHO', 'GENERO', 'GRUPO ETARIO', 'CANTIDAD', 'delito']
Poblacion   : (84233, 312) | Primeras cols: ['DP', 'DPNOM', 'MPIO', 'DPMP', 'AÑO', 'ÁREA GEOGRÁFICA', 'TOTAL Total', 'TOTAL Hombres', 'TOTAL Mujeres', 'HOMBRES Hombres 0 años', 'HOMBRES Hombres 1 año', 'HOMBRES Hombres 2 años', 'HOMBRES Hombres 3 años', 'HOMBRES Hombres 4 años', 'HOMBRES Hombres 5 años']
Forense     : (236840, 36) | Primeras cols: ['ID', 'Año del hecho', 'Sexo de la victima', 'Grupo de edad quinquenal', 'Grupo Mayor Menor de Edad', 'Grupo de Edad judicial', 'Ciclo Vital', 'País de Nacimiento', 'Escolaridad', 'Estado Civil', 'Tipo de Discapacidad', 'Pertenencia Étnica', 'Orientación Sexual', 'Identidad de Género', 'Transgénero']


In [6]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

#from loader import DataLoader
from metadata_mapper import MetadataMapper

# 1. Carga cruda
#loader = DataLoader(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
#raw = loader.load_all_datasets(['violencia', 'sexuales', 'poblacion', 'forense'])

# 2. Rutas reales
raw_paths = {
    k: loader.base_path / loader.paths.get('raw_dir', 'raw') / loader.datasets_config[k]['filename']
    for k in raw.keys()
}

# 3. Mapeo estructural (ahora blindado)
mapper = MetadataMapper(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
reporte = mapper.map_all(raw, raw_paths=raw_paths)

# 4. Visualización
mapper.print_tablero(reporte)

print("\n✅ Etapa 2 cerrada: MetadataMapper ejecutado sin errores.")

2026-06-10 10:19:41,912 - INFO - 📂 Ruta de docs validada: /Users/anaaguirre/Documents/Cicatrices_invisibles/docs
2026-06-10 10:19:41,912 - INFO - 🚀 Mapeando 5 dataset(s)...
2026-06-10 10:19:41,913 - INFO - 📸 Mapeando estructura: violencia
2026-06-10 10:19:41,916 - INFO - 📸 Mapeando estructura: sexuales
2026-06-10 10:19:41,919 - INFO - 📸 Mapeando estructura: poblacion
2026-06-10 10:19:41,999 - INFO - 📸 Mapeando estructura: forense
2026-06-10 10:19:42,003 - INFO - 📸 Mapeando estructura: seforense
2026-06-10 10:19:42,010 - INFO - 📄 Reporte exportado: /Users/anaaguirre/Documents/Cicatrices_invisibles/docs/mapeo_estructural_raw.csv


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

📊 TABLERO DE RECONOCIMIENTO ESTRUCTURAL

📦 VIOLENCIA
   📐 Estructura: 682,558 filas × 8 columnas
   💾 Memoria RAM: 82.33 MB | Disco: 60.61 MB
   🧬 Tipos: {'DEPARTAMENTO': 'string', 'MUNICIPIO': 'string', 'CODIGO DANE': 'Int64', 'ARMAS MEDIOS': 'string', 'FECHA HECHO': 'string', 'GENERO': 'string', 'GRUPO ETARIO': 'string', 'CANTIDAD': 'Int64'}
   📋 Schema: ['DEPARTAMENTO', 'MUNICIPIO', 'CODIGO DANE', 'ARMAS MEDIOS', 'FECHA HECHO']...

📦 SEXUALES
   📐 Estructura: 392,576 filas × 9 columnas
   💾 Memoria RAM: 69.61 MB | Disco: 55.24 MB
   🧬 Tipos: {'DEPARTAMENTO': 'string', 'MUNICIPIO': 'string', 'CODIGO DANE': 'Int64', 'ARMAS MEDIOS': 'string', 'FECHA HECHO': 'string', 'GENERO': 'string', 'GRUPO ETARIO': 'string', 'CANTIDAD': 'Int64', 'delito': 'string'}
   📋 Schema: ['DEPARTAMENTO', 'MUNICIPIO', 'CODIGO DANE', 'ARMAS MEDIOS', 'FECHA HECHO']...

📦 POBLACION
   📐 Estructura: 84,233 filas × 312 columna

In [7]:
import sys
from pathlib import Path

%load_ext autoreload
%autoreload 2

# 🔒 Seguridad: verifica que 'raw' exista en memoria (viene de la Celda 1)
if 'raw' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la Celda 1 (DataLoader). 'raw' no está en memoria.")

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_standardizer import DataStandardizer

# 1. Ejecutar estandarización
print("🔧 Iniciando estandarización de los datasets...")
std = DataStandardizer(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
standardized = std.standardize_all(raw)

# 2. Validación detallada y defensiva
print("\n✅ CELDA 3 - DATA STANDARDIZER EJECUTADO")
print("="*60)

for name, df in standardized.items():
    print(f"\n📦 {name.upper()}")
    print(f"   Filas: {len(df):,} | Columnas: {len(df.columns)}")

    claves = [c for c in ['fecha_hecho', 'cod_municipio', 'departamento'] if c in df.columns]
    print(f"   📌 Claves críticas: {claves}")

    if 'fecha_hecho' in df.columns:
        dtype = df['fecha_hecho'].dtype
        nulos = df['fecha_hecho'].isna().sum()
        muestra = df['fecha_hecho'].dropna().head(2).tolist()
        print(f"   📅 fecha_hecho: {dtype} | NaT: {nulos} | Muestra: {muestra}")

    if 'departamento' in df.columns:
        unicos = [str(x) for x in df['departamento'].dropna().unique()[:5]]
        print(f"   📍 Departamentos (muestra): {unicos}")


    # 📊 DANE específico
    if name == 'poblacion':
        pob_cols = [c for c in df.columns if c.startswith('pob_')]
        print(f"   📊 Agregación DANE: {'✅ SÍ' if 'pob_f_0_17' in df.columns else '❌ NO'}")
        if pob_cols:
            print(f"      • Columnas: {pob_cols}")

print("\n🛡️ Verificación final:")
print(f"• Datasets procesados: {len(standardized)}/{len(loader.datasets_config)}")
print("✅ Etapa 3 completada. Listo para DataSegmenter.")

2026-06-10 10:19:46,950 - INFO - 🚀 Estandarizando 5 dataset(s)...
2026-06-10 10:19:46,950 - INFO - 🔧 Estandarizando: violencia (682,558 filas, 8 cols)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🔧 Iniciando estandarización de los datasets...


2026-06-10 10:19:49,018 - INFO - 🏷️ Geo-Integridad: 4 códigos inválidos → 'CODIGO_MAL_REGISTRADO'
2026-06-10 10:19:49,072 - INFO - ✅ violencia estandarizado. Columnas finales: 11
2026-06-10 10:19:49,072 - INFO - 🔧 Estandarizando: sexuales (392,576 filas, 9 cols)
2026-06-10 10:19:50,259 - INFO - 🏷️ Geo-Integridad: 3 códigos inválidos → 'CODIGO_MAL_REGISTRADO'
2026-06-10 10:19:50,646 - INFO - ✅ sexuales estandarizado. Columnas finales: 13
2026-06-10 10:19:50,646 - INFO - 🔧 Estandarizando: poblacion (84,233 filas, 312 cols)
2026-06-10 10:19:51,334 - INFO - ✅ poblacion estandarizado. Columnas finales: 115
2026-06-10 10:19:51,334 - INFO - 🔧 Estandarizando: forense (236,840 filas, 36 cols)
2026-06-10 10:19:54,972 - INFO - ✅ forense estandarizado. Columnas finales: 41
2026-06-10 10:19:54,973 - INFO - 🔧 Estandarizando: seforense (233,352 filas, 32 cols)
2026-06-10 10:19:58,291 - INFO - ✅ seforense estandarizado. Columnas finales: 38



✅ CELDA 3 - DATA STANDARDIZER EJECUTADO

📦 VIOLENCIA
   Filas: 682,558 | Columnas: 11
   📌 Claves críticas: ['fecha_hecho', 'cod_municipio', 'departamento']
   📅 fecha_hecho: datetime64[us] | NaT: 0 | Muestra: [Timestamp('2026-03-10 00:00:00'), Timestamp('2026-03-15 00:00:00')]
   📍 Departamentos (muestra): ['CALDAS', 'ATLANTICO', 'HUILA', 'CUNDINAMARCA', 'VALLE DEL CAUCA']

📦 SEXUALES
   Filas: 392,576 | Columnas: 13
   📌 Claves críticas: ['fecha_hecho', 'cod_municipio', 'departamento']
   📅 fecha_hecho: datetime64[us] | NaT: 0 | Muestra: [Timestamp('2010-02-04 00:00:00'), Timestamp('2010-02-23 00:00:00')]
   📍 Departamentos (muestra): ['PUTUMAYO', 'CASANARE', 'VALLE DEL CAUCA', 'CORDOBA', 'GUAJIRA']

📦 POBLACION
   Filas: 84,233 | Columnas: 115
   📌 Claves críticas: ['fecha_hecho', 'cod_municipio', 'departamento']
   📅 fecha_hecho: datetime64[ns] | NaT: 8 | Muestra: [Timestamp('2018-01-01 00:00:00'), Timestamp('2018-01-01 00:00:00')]
   📍 Departamentos (muestra): ['ANTIOQUIA', 'ATLA

In [8]:
import sys
import json
from pathlib import Path

%load_ext autoreload
%autoreload 2

if 'standardized' not in globals():
    raise RuntimeError("⚠️ Ejecuta primero la Etapa 3 (DataStandardizer).")

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ✅ CAMBIO AQUÍ: Fusionar base_config.json + mapping_config.json
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f:
        base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}  # El específico gana prioridad

# ✅ Ahora sí encuentra display_config (vive en base_config.json)
disp = config.get('display_config')
if not disp: raise ValueError("❌ 'display_config' no definido en la configuración fusionada")

SEP = disp['separator_char'] * disp['separator_length']
IND = disp['indent']
PREFIX_DS = disp['prefix_dataset']
TITLE = disp['section_title'].format(n=4, module="DATA SEGMENTER")

from data_segmenter import DataSegmenter

print(f"✂️ SEGMENTACIÓN VECTORIAL (Departamento Y Año 2018-2025)")
print(SEP)

segmenter = DataSegmenter(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
metrics_df = segmenter.segment_all(standardized)

print(f"\n{TITLE}")
print(SEP)

for _, row in metrics_df.iterrows():
    print(f"\n{PREFIX_DS}{row['dataset'].upper()}")
    print(f"{IND}📥 Originales: {row['filas_originales']:,}")
    print(f"{IND}✅ Cumplen AMBOS (Depto + Año): {row['filas_cumplen_ambos']:,}")
    print(f"{IND}⚠️  Excluidas (fuera de alcance): {row['excluidas_no_cumplen']:,}")
    print(f"{IND}📊 Columnas: {row['columnas_preservadas']} (100% conservadas)")
    print(f"{IND}💾 Guardado en: data/segmented/{Path(row['ruta_parquet']).name}")

print(f"\n🛡️ Garantías aplicadas:")
print(f"{IND}• Filtrado 100% vectorial: mask = df['depto'].isin() & df['año'].between()")
print(f"{IND}• Lógica AND estricta: solo entran filas con departamento válido Y año 2018-2025.")
print(f"{IND}• NaT en fecha_hecho → excluido nativamente (no cumple rango temporal).")
print(f"{IND}• Rutas dinámicas: combina base_data_dir + segmented_dir desde JSON. Cero carpetas huérfanas.")

print(SEP)
print(f"✅ Etapa 4 completada. Listo para DataAuditor.")

2026-06-10 10:20:02,938 - INFO - 📂 Usando carpeta configurada: /Users/anaaguirre/Documents/Cicatrices_invisibles/data/segmented
2026-06-10 10:20:02,938 - INFO - ✂️ Segmentando (AND Vectorial): Depto + Año 2018-2025


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✂️ SEGMENTACIÓN VECTORIAL (Departamento Y Año 2018-2025)

✅ CELDA 4 - DATA SEGMENTER EJECUTADO

📦 VIOLENCIA
   📥 Originales: 682,558
   ✅ Cumplen AMBOS (Depto + Año): 68,073
   ⚠️  Excluidas (fuera de alcance): 614,485
   📊 Columnas: 11 (100% conservadas)
   💾 Guardado en: data/segmented/violencia_2018_2025.parquet

📦 SEXUALES
   📥 Originales: 392,576
   ✅ Cumplen AMBOS (Depto + Año): 37,802
   ⚠️  Excluidas (fuera de alcance): 354,774
   📊 Columnas: 13 (100% conservadas)
   💾 Guardado en: data/segmented/sexuales_2018_2025.parquet

📦 POBLACION
   📥 Originales: 84,233
   ✅ Cumplen AMBOS (Depto + Año): 4,296
   ⚠️  Excluidas (fuera de alcance): 79,937
   📊 Columnas: 115 (100% conservadas)
   💾 Guardado en: data/segmented/poblacion_2018_2025.parquet

📦 FORENSE
   📥 Originales: 236,840
   ✅ Cumplen AMBOS (Depto + Año): 15,418
   ⚠️  Excluidas (fuera de alcance): 221,422
   📊 Columnas: 41 (100% conservad

In [9]:
import sys
import json
from pathlib import Path
import pandas as pd

%load_ext autoreload
%autoreload 2

# 🔒 Anclaje determinista a raíz del proyecto
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ✅ CAMBIO AQUÍ: Fusión base + mapping para recuperar 'paths' y 'display_config'
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}  # paths y display_config se inyectan desde base

# ✅ Rutas resueltas ABSOLUTAMENTE desde JSON + project_root
SEGMENTED_DIR = (PROJECT_ROOT / config['paths']['segmented_dir']).resolve()
DOCS_DIR = (PROJECT_ROOT / config['paths']['docs_dir']).resolve()

if not SEGMENTED_DIR.exists():
    raise RuntimeError(f"⚠️ Carpeta segmentada no encontrada: {SEGMENTED_DIR}. Ejecuta Etapa 4.")
if not DOCS_DIR.exists():
    raise FileNotFoundError(f"❌ Carpeta docs no encontrada: {DOCS_DIR}. Verifica estructura.")

disp = config.get('display_config')
if not disp: raise ValueError("❌ 'display_config' no definido en JSON")

SEP = disp['separator_char'] * disp['separator_length']
IND = disp['indent']
PREFIX_DS = disp['prefix_dataset']
TITLE = disp['section_title'].format(n=5, module="DATA AUDITOR")

# ✅ Importación exacta (archivo debe estar en src/data_auditor.py)
from data_auditor import DataAuditor

print(f"🔍 AUDITORÍA INTEGRAL (100% Columnas | 100% Filas)")
print(SEP)

segmented_data = {f.stem.split('_')[0]: pd.read_parquet(f) for f in SEGMENTED_DIR.glob("*.parquet")}
auditor = DataAuditor(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
summary_df = auditor.audit_all(segmented_data)

print(f"\n{TITLE}")
print(SEP)

for _, row in summary_df.iterrows():
    print(f"\n{PREFIX_DS}{row['dataset'].upper()}")
    print(f"{IND}📊 Filas auditadas: {row['rows']:,}")
    print(f"{IND}⚠️  Columnas con nulos: {row['cols_with_nulls']}")
    print(f"{IND}🪞 Duplicados espejo: {row['mirror_duplicates']:,}")
    print(f"{IND}🧬 Schema issues: {row['schema_issues']}")
    print(f"{IND}🌐 Dominio issues: {row['domain_violations']}")
    print(f"{IND}📅 Temporal issues: {row['temporal_issues']}")
    print(f"{IND}🔢 Numeric issues: {row['numeric_issues']}")
    print(f"{IND}📄 Reporte: docs/{Path(row['report_path']).name}")

print(f"\n🛡️ Garantías:")
print(f"{IND}• Rutas ancladas a project_root: cero carpetas duplicadas")
print(f"{IND}• 100% vectorial + dominio por columna + .astype('string') seguro")
print(f"{IND}• Reportes guardados en tu docs/ existente")
print(SEP)
print(f"✅ Etapa 5 completada.")

2026-06-10 10:20:13,959 - INFO - 📂 Ruta de reporte validada: /Users/anaaguirre/Documents/Cicatrices_invisibles/docs
2026-06-10 10:20:13,960 - INFO - 🚀 Auditando 5 dataset(s) de forma integral...
2026-06-10 10:20:13,960 - INFO - 🔍 Auditando: violencia (68,073 filas)
2026-06-10 10:20:13,981 - INFO - 🔍 Auditando: forense (15,418 filas)
2026-06-10 10:20:13,996 - INFO - 🔍 Auditando: sexuales (37,802 filas)
2026-06-10 10:20:14,008 - INFO - 🔍 Auditando: poblacion (4,296 filas)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🔍 AUDITORÍA INTEGRAL (100% Columnas | 100% Filas)


2026-06-10 10:20:14,031 - INFO - 🔍 Auditando: seforense (18,661 filas)



✅ CELDA 5 - DATA AUDITOR EJECUTADO

📦 VIOLENCIA
   📊 Filas auditadas: 68,073
   ⚠️  Columnas con nulos: 2
   🪞 Duplicados espejo: 21
   🧬 Schema issues: 0
   🌐 Dominio issues: 0
   📅 Temporal issues: 0
   🔢 Numeric issues: 1
   📄 Reporte: docs/auditoria_violencia.csv

📦 FORENSE
   📊 Filas auditadas: 15,418
   ⚠️  Columnas con nulos: 0
   🪞 Duplicados espejo: 0
   🧬 Schema issues: 0
   🌐 Dominio issues: 0
   📅 Temporal issues: 0
   🔢 Numeric issues: 0
   📄 Reporte: docs/auditoria_forense.csv

📦 SEXUALES
   📊 Filas auditadas: 37,802
   ⚠️  Columnas con nulos: 2
   🪞 Duplicados espejo: 3,210
   🧬 Schema issues: 0
   🌐 Dominio issues: 0
   📅 Temporal issues: 0
   🔢 Numeric issues: 0
   📄 Reporte: docs/auditoria_sexuales.csv

📦 POBLACION
   📊 Filas auditadas: 4,296
   ⚠️  Columnas con nulos: 0
   🪞 Duplicados espejo: 0
   🧬 Schema issues: 0
   🌐 Dominio issues: 0
   📅 Temporal issues: 0
   🔢 Numeric issues: 6
   📄 Reporte: docs/auditoria_poblacion.csv

📦 SEFORENSE
   📊 Filas auditadas: 18,

In [12]:
import sys
import json
from pathlib import Path
import pandas as pd

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))

# ✅ CAMBIO: Fusión base + mapping para recuperar 'paths' y 'display_config'
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}  # paths y display_config se inyectan desde base

disp = config.get('display_config')
if not disp: raise ValueError("❌ 'display_config' no definido")
SEP = disp['separator_char'] * disp['separator_length']
IND = disp['indent']
PREFIX_DS = disp['prefix_dataset']
TITLE = disp['section_title'].format(n=6, module="DATA CLEANER")

from data_cleaner import DataCleaner

print(f"🧼 LIMPIEZA QUIRÚRGICA (JSON-Driven | Vectorial)")
print(SEP)

# Cargar segmentados
SEGMENTED_DIR = (PROJECT_ROOT / config['paths']['segmented_dir']).resolve()
segmented_data = {f.stem.split('_')[0]: pd.read_parquet(f) for f in SEGMENTED_DIR.glob("*.parquet")}

cleaner = DataCleaner(config_path=str(PROJECT_ROOT / "config" / "mapping_config.json"))
metrics_df = cleaner.clean_all(segmented_data)

print(f"\n{TITLE}")
print(SEP)

for _, row in metrics_df.iterrows():
    print(f"\n{PREFIX_DS}{row['dataset'].upper()}")
    print(f"{IND}📥 Entrada: {row['filas_entrada']:,} filas")
    print(f"{IND}📤 Salida limpia: {row['filas_salida']:,} filas")
    print(f"{IND}🚫 Duplicados eliminados: {row['duplicados_eliminados']:,}")
    print(f"{IND}💊 Nulos imputados: {row['nulos_imputados']:,}")
    print(f"{IND}💾 Guardado: data/cleaned/{Path(row['ruta_limpio']).name}")

print(f"\n🛡️ Garantías:")
print(f"{IND}• Deduplicación espejo con cuarentena en docs/")
print(f"{IND}• Imputación vectorial SOLO en campos críticos (JSON)")
print(f"{IND}• Archivos originales en segmented/ intactos")
print(SEP)
print(f"✅ Etapa 6 completada. Datos listos para MasterBuilder.")

2026-06-10 10:28:45,453 - INFO - 🚀 Iniciando limpieza de 5 dataset(s)...
2026-06-10 10:28:45,453 - INFO - 🧼 Limpiando: violencia (68,073 filas)
2026-06-10 10:28:45,465 - INFO - 🚫 21 duplicados espejo → cuarentena: cuarentena_duplicados_violencia.csv
2026-06-10 10:28:45,491 - INFO - 🧼 Limpiando: forense (15,418 filas)
2026-06-10 10:28:45,511 - INFO - 🧼 Limpiando: sexuales (37,802 filas)
2026-06-10 10:28:45,527 - INFO - 🚫 3210 duplicados espejo → cuarentena: cuarentena_duplicados_sexuales.csv
2026-06-10 10:28:45,541 - INFO - 🧼 Limpiando: poblacion (4,296 filas)
2026-06-10 10:28:45,558 - INFO - 🧼 Limpiando: seforense (18,661 filas)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
🧼 LIMPIEZA QUIRÚRGICA (JSON-Driven | Vectorial)

✅ CELDA 6 - DATA CLEANER EJECUTADO

📦 VIOLENCIA
   📥 Entrada: 68,073 filas
   📤 Salida limpia: 68,052 filas
   🚫 Duplicados eliminados: 21
   💊 Nulos imputados: 383
   💾 Guardado: data/cleaned/violencia_limpio.parquet

📦 FORENSE
   📥 Entrada: 15,418 filas
   📤 Salida limpia: 15,418 filas
   🚫 Duplicados eliminados: 0
   💊 Nulos imputados: 0
   💾 Guardado: data/cleaned/forense_limpio.parquet

📦 SEXUALES
   📥 Entrada: 37,802 filas
   📤 Salida limpia: 34,592 filas
   🚫 Duplicados eliminados: 3,210
   💊 Nulos imputados: 189
   💾 Guardado: data/cleaned/sexuales_limpio.parquet

📦 POBLACION
   📥 Entrada: 4,296 filas
   📤 Salida limpia: 4,296 filas
   🚫 Duplicados eliminados: 0
   💊 Nulos imputados: 0
   💾 Guardado: data/cleaned/poblacion_limpio.parquet

📦 SEFORENSE
   📥 Entrada: 18,661 filas
   📤 Salida limpia: 18,661 filas
   🚫 Duplicados eliminados: 0
   💊

In [13]:
import pandas as pd, json
from pathlib import Path

# 🔒 Misma lógica de resolución de rutas que usa el pipeline
PROJECT_ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()).lower() else Path.cwd()

# ✅ CAMBIO: Fusión base + mapping para recuperar 'paths'
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", encoding='utf-8') as f:
    spec_cfg = json.load(f)
cfg = {**base_cfg, **spec_cfg}  # paths se inyecta desde base

CLEAN_DIR = (PROJECT_ROOT / cfg["paths"]["cleaned_dir"]).resolve()

print("🔍 VERIFICACIÓN DE COLUMNAS POST-CLEANER\n" + "="*60)
for ds in ["violencia", "sexuales", "forense", "seforense", "poblacion"]:
    fpath = CLEAN_DIR / f"{ds}_limpio.parquet"
    if fpath.exists():
        df = pd.read_parquet(fpath)
        print(f"📊 {ds.upper():<10}: {len(df):,} filas | {len(df.columns)} cols")
        print(f"   🔹 {df.columns.tolist()}\n")
    else:
        print(f"❌ {ds.upper():<10}: No encontrado en {fpath.name}\n")

🔍 VERIFICACIÓN DE COLUMNAS POST-CLEANER
📊 VIOLENCIA : 68,052 filas | 10 cols
   🔹 ['departamento', 'municipio', 'cod_municipio', 'fecha_hecho', 'anio_hecho', 'mes_hecho', 'dia_hecho', 'genero_victima', 'grupo_etario', 'cantidad']

📊 SEXUALES  : 34,592 filas | 11 cols
   🔹 ['departamento', 'municipio', 'cod_municipio', 'fecha_hecho', 'anio_hecho', 'mes_hecho', 'dia_hecho', 'genero_victima', 'grupo_etario', 'cantidad', 'dimension_delito']

📊 FORENSE   : 15,418 filas | 21 cols
   🔹 ['anio_hecho', 'genero_victima', 'ciclo_vital', 'grupo_edad', 'cod_municipio', 'municipio', 'departamento', 'escenario', 'agresor', 'sexo_agresor', 'factor', 'discapacidad', 'etnia', 'hora_rango', 'mes_hecho', 'dia_hecho', 'dias_incapacidad', 'dimension_agresor', 'dimension_escenario', 'dimension_discapacidad', 'dimension_etnia']

📊 SEFORENSE : 18,661 filas | 21 cols
   🔹 ['anio_hecho', 'genero_victima', 'grupo_edad', 'ciclo_vital', 'discapacidad', 'etnia', 'mes_hecho', 'dia_hecho', 'hora_rango', 'cod_municipio

In [14]:
import sys, json
from pathlib import Path
%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))

# ✅ CAMBIO: Fusión base + master_builder_config.json
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "master_builder_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}  # display_config viene de base, aggregation de master_builder

disp = config['display_config']  # ✅ Ahora se resuelve desde la fusión
SEP = disp['separator_char'] * disp['separator_length']
TITLE = disp['section_title'].format(n=6, module="DATA AGGREGATOR")

from data_aggregator import DataAggregator
# ✅ IMPORTANTE: Apuntar al JSON donde vive aggregation_config
agg = DataAggregator(config_path=str(PROJECT_ROOT / "config" / "master_builder_config.json"))
layers_ram = agg.aggregate_all()

print(f"\n{TITLE}\n{SEP}")
for name, df in layers_ram.items():
    print(f"🔹 {name}: {len(df):,} filas")
    print(f"   📊 Columnas: {list(df.columns)}")
    print(f"   🔍 Ejemplo: {df.iloc[0].to_dict() if len(df) > 0 else 'Vacío'}\n")
print(f"✅ Todo en RAM. Sin escritura en disco. Listo para esqueleto + joins.")
print(SEP)

2026-06-10 10:30:32,436 - INFO - 📐 Construyendo 8 capas analíticas en RAM...
2026-06-10 10:30:32,448 - INFO - ✅ casos_vif_nna_f: 523 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-10 10:30:32,462 - INFO - ✅ casos_vif_adultas_f: 1,327 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-10 10:30:32,469 - INFO - ✅ casos_sexual_nna_f: 1,024 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-10 10:30:32,479 - INFO - ✅ casos_sexual_adultas_f: 1,272 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-10 10:30:32,486 - INFO - ✅ casos_vif_nna_m: 372 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-10 10:30:32,495 - INFO - ✅ casos_vif_adultos_m: 1,021 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-10 10:30:32,501 - INFO - ✅ casos_sexual_nna_m: 416 filas | columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
2026-06-10 10:30:32,508 - INFO - ✅ casos_sexual_adultos_m: 608 filas

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

✅ CELDA 6 - DATA AGGREGATOR EJECUTADO
🔹 casos_vif_nna_f: 523 filas
   📊 Columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
   🔍 Ejemplo: {'cod_municipio': '19001', 'anio_hecho': 2018, 'cantidad': 47}

🔹 casos_vif_adultas_f: 1,327 filas
   📊 Columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
   🔍 Ejemplo: {'cod_municipio': '19001', 'anio_hecho': 2018, 'cantidad': 1181}

🔹 casos_sexual_nna_f: 1,024 filas
   📊 Columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
   🔍 Ejemplo: {'cod_municipio': '19001', 'anio_hecho': 2018, 'cantidad': 118}

🔹 casos_sexual_adultas_f: 1,272 filas
   📊 Columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
   🔍 Ejemplo: {'cod_municipio': '19001', 'anio_hecho': 2018, 'cantidad': 215}

🔹 casos_vif_nna_m: 372 filas
   📊 Columnas: ['cod_municipio', 'anio_hecho', 'cantidad']
   🔍 Ejemplo: {'cod_municipio': '19001', 'anio_hecho': 2018, 'cantidad': 27}

🔹 casos_vif_adultos

In [15]:
import sys, json, pandas as pd
from pathlib import Path
%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))

# ✅ 1. Verificación de dependencia de memoria (viene de la celda de agregación)
if 'layers_ram' not in globals():
    raise RuntimeError("⚠️ 'layers_ram' no encontrado en RAM. Ejecuta primero la celda del DataAggregator en este mismo kernel.")

# ✅ 2. Carga temporal de limpios para comparación (diccionario en RAM, NO carpeta)
CLEAN_DIR = (PROJECT_ROOT / "data/cleaned").resolve()
cleaned_datasets = {
    "violencia": pd.read_parquet(CLEAN_DIR / "violencia_limpio.parquet"),
    "sexuales": pd.read_parquet(CLEAN_DIR / "sexuales_limpio.parquet")
}

# ✅ 3. Carga de config para formato (100% desde JSON)
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "master_builder_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}

disp = config.get('display_config', {})
SEP = disp.get('separator_char', '=') * disp.get('separator_length', 70)
IND = disp.get('indent', '   ')
PREFIX = disp.get('prefix_dataset', '📦 ')
TITLE = disp.get('section_title', '✅ CELDA {n} - {module} EJECUTADO').format(n=8, module="VALIDACIÓN CAPAS")

from validator import LayerValidator

print(f"\n{TITLE}")
print(SEP)

validator = LayerValidator(config_path=str(PROJECT_ROOT / "config" / "master_builder_config.json"))
report = validator.validate(layers_ram, cleaned_datasets)

print(f"\n{IND}📊 Resultados:")
for check in report['checks']:
    icon = "✅" if check['status'] == 'PASS' else "❌"
    print(f"{IND}{PREFIX}{icon} {check['check']:<25} | {check.get('group', check.get('layer', '')):<15} | {check['msg']}")

print(SEP)
print(f"{IND}🛡️ Estado global: {report['status']}")
print(f"{IND}📄 Reporte trazable: docs/{validator.output_filename}")
print(f"{IND}✅ Capas validadas. Listo para Esqueleto Territorial + MasterBuilder.")
print(SEP)

2026-06-10 10:31:17,803 - INFO - 📄 Reporte exportado: /Users/anaaguirre/Documents/Cicatrices_invisibles/docs/validacion_capas_pre_join.csv
2026-06-10 10:31:17,803 - INFO - ✅ Validación cruzada de capas: PASS. Pipeline continuando...


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

✅ CELDA 8 - VALIDACIÓN CAPAS EJECUTADO

   📊 Resultados:
   📦 ✅ SUMA_TOTALES              | violencia_total | Suma violencia_total: 122825 vs 122825 (diff: 0.00%)
   📦 ✅ SUMA_TOTALES              | sexuales_total  | Suma sexuales_total: 34446 vs 34446 (diff: 0.00%)
   📦 ✅ UNICIDAD                  | casos_vif_nna_f | casos_vif_nna_f: Unicidad garantizada
   📦 ✅ COBERTURA_TEMPORAL        | casos_vif_nna_f | casos_vif_nna_f: Cobertura completa (2018-2025)
   📦 ✅ UNICIDAD                  | casos_vif_adultas_f | casos_vif_adultas_f: Unicidad garantizada
   📦 ✅ COBERTURA_TEMPORAL        | casos_vif_adultas_f | casos_vif_adultas_f: Cobertura completa (2018-2025)
   📦 ✅ UNICIDAD                  | casos_sexual_nna_f | casos_sexual_nna_f: Unicidad garantizada
   📦 ✅ COBERTURA_TEMPORAL        | casos_sexual_nna_f | casos_sexual_nna_f: Cobertura completa (2018-2025)
   📦 ✅ UNICIDAD                  | casos_

In [16]:
import sys, json, pandas as pd
from pathlib import Path
%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path: sys.path.insert(0, str(SRC_DIR))

if 'layers_ram' not in globals():
    raise RuntimeError("⚠️ 'layers_ram' no encontrado. Ejecuta agregación y validación primero.")

CLEAN_DIR = (PROJECT_ROOT / "data/cleaned").resolve()
dane_df = pd.read_parquet(CLEAN_DIR / "poblacion_limpio.parquet")

base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "master_builder_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
config = {**base_cfg, **spec_cfg}

disp = config.get('display_config', {})
SEP = disp.get('separator_char', '=') * disp.get('separator_length', 70)
IND = disp.get('indent', '   ')
PREFIX = disp.get('prefix_dataset', '📦 ')
TITLE = disp.get('section_title', '✅ CELDA {n} - {module} EJECUTADO').format(n=9, module="DATA MASTER BUILDER")

from masterbuilder import DataMasterBuilder

print(f"\n{TITLE}")
print(SEP)

builder = DataMasterBuilder(config_path=str(PROJECT_ROOT / "config" / "master_builder_config.json"))
master_df = builder.build(layers_ram, dane_df)

print(f"\n{IND}📊 Resumen Tabla Maestra:")
print(f"{IND}{PREFIX} Filas maestro anual: {len(master_df):,} | Columnas: {len(master_df.columns)}")
print(f"{IND}{PREFIX} Filas tabla clustering: {pd.read_parquet(builder.master_dir / builder.output_cfg['clustering_filename']).shape[0]}")
print(f"{IND}{PREFIX} Rango ICV-GEN-F: {master_df['icv_gen_f_score'].min():.2f} - {master_df['icv_gen_f_score'].max():.2f}")
print(SEP)
print(f"{IND}🛡️ Garantías aplicadas:")
print(f"{IND}• Esqueleto DANE × años (NaN en casos → 0 | NaN en población → preservado)")
print(f"{IND}• Brechas f/m con manejo seguro de división por cero → NaN")
print(f"{IND}• ICV-GEN-F normalizado Min-Max y ponderado por JSON")
print(f"{IND}• Tabla de clustering colapsada por municipio para K-Means")
print(SEP)
print(f"✅ Etapa 9 completada. Pipeline cerrado. Listo para EDA + ML.")

2026-06-10 10:31:51,188 - INFO - 🏗️ Construyendo Tabla Maestra (Esqueleto + JOINs + Tasas + Brechas + ICV)...
2026-06-10 10:31:51,225 - INFO - 💾 Maestro anual exportado: /Users/anaaguirre/Documents/Cicatrices_invisibles/data/master/master_table.parquet (4,296 filas)
2026-06-10 10:31:51,231 - INFO - 💾 Tabla Clustering exportada: /Users/anaaguirre/Documents/Cicatrices_invisibles/data/master/tabla_clustering.parquet (179 filas)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

✅ CELDA 9 - DATA MASTER BUILDER EJECUTADO

   📊 Resumen Tabla Maestra:
   📦  Filas maestro anual: 4,296 | Columnas: 23
   📦  Filas tabla clustering: 179
   📦  Rango ICV-GEN-F: 0.00 - 91.17
   🛡️ Garantías aplicadas:
   • Esqueleto DANE × años (NaN en casos → 0 | NaN en población → preservado)
   • Brechas f/m con manejo seguro de división por cero → NaN
   • ICV-GEN-F normalizado Min-Max y ponderado por JSON
   • Tabla de clustering colapsada por municipio para K-Means
✅ Etapa 9 completada. Pipeline cerrado. Listo para EDA + ML.


###**Revision DSets**

In [17]:
import pandas as pd
from pathlib import Path

# 🔒 Detección de raíz igual que tu pipeline
PROJECT_ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()).lower() else Path.cwd()
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"

datasets = ["violencia", "sexuales", "poblacion", "forense", "seforense"]

for ds in datasets:
    fpath = CLEAN_DIR / f"{ds}_limpio.parquet"
    if fpath.exists():
        # Lee solo metadatos de columnas (cero RAM innecesaria)
        cols = pd.read_parquet(fpath).columns.tolist()
        print(f"📊 {ds.upper()} ({len(cols)} columnas):")
        for c in cols:
            print(f"   - {c}")
        print("-" * 60)
    else:
        print(f"❌ {ds.upper()}: No existe en {CLEAN_DIR}")

📊 VIOLENCIA (10 columnas):
   - departamento
   - municipio
   - cod_municipio
   - fecha_hecho
   - anio_hecho
   - mes_hecho
   - dia_hecho
   - genero_victima
   - grupo_etario
   - cantidad
------------------------------------------------------------
📊 SEXUALES (11 columnas):
   - departamento
   - municipio
   - cod_municipio
   - fecha_hecho
   - anio_hecho
   - mes_hecho
   - dia_hecho
   - genero_victima
   - grupo_etario
   - cantidad
   - dimension_delito
------------------------------------------------------------
📊 POBLACION (11 columnas):
   - departamento
   - cod_municipio
   - anio_hecho
   - area_geo
   - municipio
   - pob_f_0_17
   - pob_f_18_mas
   - pob_h_0_17
   - pob_h_18_mas
   - total_h
   - total_f
------------------------------------------------------------
📊 FORENSE (21 columnas):
   - anio_hecho
   - genero_victima
   - ciclo_vital
   - grupo_edad
   - cod_municipio
   - municipio
   - departamento
   - escenario
   - agresor
   - sexo_agresor
   - factor
 

In [20]:
import pandas as pd, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()).lower() else Path.cwd()

# ✅ CAMBIO: Fusión base + mapping para recuperar 'paths'
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
cfg = {**base_cfg, **spec_cfg}  # paths se inyecta desde base

CLEAN_DIR = (PROJECT_ROOT / cfg["paths"]["cleaned_dir"]).resolve()
df = pd.read_parquet(CLEAN_DIR / "sexuales_limpio.parquet")

print(f"✅ DS2 SEXUALES: {len(df):,} filas | {len(df.columns)} columnas\n")
print("=" * 80)

for col in df.columns:
    uniq = df[col].dropna()
    print(f"📊 {col.upper():<30} | Tipo: {str(df[col].dtype):<12} | Únicos: {uniq.nunique():<5} | Vacíos: {df[col].isna().sum():,}")

    if len(uniq) > 0:
        top = uniq.value_counts().head(30)
        for val, cnt in top.items():
            print(f"   • {str(val):<50} → {cnt:,}")

        # ✅ Solo busca "sin info" en columnas de texto/categoría
        if df[col].dtype == 'object' or str(df[col].dtype) == 'string' or str(df[col].dtype) == 'category':
            try:
                # Regex con grupos sin captura (?:) para evitar warning
                sin_pattern = r"(?:sin informaci[oó]n|no aplica|no reportado|no registrado|n/a|na|null|vac[ií]o|s/i)"
                sin_info = uniq[uniq.astype(str).str.contains(sin_pattern, case=False, na=False, regex=True)]
                if 0 < len(sin_info) < len(uniq) * 0.95:
                    print(f"   ⚠️  Posibles 'sin info': {sin_info.value_counts().head(3).to_dict()}")
            except:
                pass
    print("-" * 80)

✅ DS2 SEXUALES: 34,592 filas | 11 columnas

📊 DEPARTAMENTO                   | Tipo: str          | Únicos: 4     | Vacíos: 0
   • VALLE DEL CAUCA                                    → 19,725
   • NARIÑO                                             → 6,932
   • CAUCA                                              → 6,316
   • CHOCO                                              → 1,619
--------------------------------------------------------------------------------
📊 MUNICIPIO                      | Tipo: str          | Únicos: 175   | Vacíos: 0
   • CALI                                               → 9,738
   • PASTO                                              → 3,043
   • POPAYAN                                            → 2,179
   • PALMIRA                                            → 1,263
   • TULUA                                              → 1,103
   • BUENAVENTURA                                       → 810
   • IPIALES                                            → 797
   • JAMUN

In [21]:
import pandas as pd, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()).lower() else Path.cwd()

# ✅ CAMBIO: Fusión base + mapping para recuperar 'paths'
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
cfg = {**base_cfg, **spec_cfg}  # paths se inyecta desde base

CLEAN_DIR = (PROJECT_ROOT / cfg["paths"]["cleaned_dir"]).resolve()
df = pd.read_parquet(CLEAN_DIR / "violencia_limpio.parquet")

print(f"✅ DS1 VIOLENCIA: {len(df):,} filas | {len(df.columns)} columnas\n")
print("=" * 80)

for col in df.columns:
    uniq = df[col].dropna()
    print(f"📊 {col.upper():<30} | Tipo: {str(df[col].dtype):<12} | Únicos: {uniq.nunique():<5} | Vacíos: {df[col].isna().sum():,}")

    if len(uniq) > 0:
        top = uniq.value_counts().head(30)
        for val, cnt in top.items():
            print(f"   • {str(val):<50} → {cnt:,}")

        # ✅ Solo busca "sin info" en columnas de texto/categoría
        if df[col].dtype == 'object' or str(df[col].dtype) == 'string' or str(df[col].dtype) == 'category':
            try:
                # Regex con grupos sin captura (?:) para evitar warning
                sin_pattern = r"(?:sin informaci[oó]n|no aplica|no reportado|no registrado|n/a|na|null|vac[ií]o|s/i)"
                sin_info = uniq[uniq.astype(str).str.contains(sin_pattern, case=False, na=False, regex=True)]
                if 0 < len(sin_info) < len(uniq) * 0.95:
                    print(f"   ⚠️  Posibles 'sin info': {sin_info.value_counts().head(3).to_dict()}")
            except:
                pass
    print("-" * 80)

✅ DS1 VIOLENCIA: 68,052 filas | 10 columnas

📊 DEPARTAMENTO                   | Tipo: str          | Únicos: 4     | Vacíos: 0
   • VALLE DEL CAUCA                                    → 36,252
   • CAUCA                                              → 15,021
   • NARIÑO                                             → 14,442
   • CHOCO                                              → 2,337
--------------------------------------------------------------------------------
📊 MUNICIPIO                      | Tipo: str          | Únicos: 174   | Vacíos: 0
   • CALI                                               → 12,402
   • PASTO                                              → 7,297
   • POPAYAN                                            → 5,964
   • PALMIRA                                            → 3,079
   • GUADALAJARA DE BUGA                                → 2,108
   • TULUA                                              → 2,042
   • YUMBO                                              → 2,031
  

In [23]:
import pandas as pd, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()).lower() else Path.cwd()

# ✅ CAMBIO: Fusión base + mapping para recuperar 'paths'
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
cfg = {**base_cfg, **spec_cfg}  # paths se inyecta desde base

CLEAN_DIR = (PROJECT_ROOT / cfg["paths"]["cleaned_dir"]).resolve()
df = pd.read_parquet(CLEAN_DIR / "seforense_limpio.parquet")

print(f"✅ DS3 SEXUAL FORENSE: {len(df):,} filas | {len(df.columns)} columnas\n")
print("=" * 80)

for col in df.columns:
    uniq = df[col].dropna()
    print(f"📊 {col.upper():<30} | Tipo: {str(df[col].dtype):<12} | Únicos: {uniq.nunique():<5} | Vacíos: {df[col].isna().sum():,}")

    if len(uniq) > 0:
        top = uniq.value_counts().head(30)
        for val, cnt in top.items():
            print(f"   • {str(val):<50} → {cnt:,}")

        # ✅ Solo busca "sin info" en columnas de texto/categoría
        if df[col].dtype == 'object' or str(df[col].dtype) == 'string' or str(df[col].dtype) == 'category':
            try:
                # Regex con grupos sin captura (?:) para evitar warning
                sin_pattern = r"(?:sin informaci[oó]n|no aplica|no reportado|no registrado|n/a|na|null|vac[ií]o|s/i)"
                sin_info = uniq[uniq.astype(str).str.contains(sin_pattern, case=False, na=False, regex=True)]
                if 0 < len(sin_info) < len(uniq) * 0.95:
                    print(f"   ⚠️  Posibles 'sin info': {sin_info.value_counts().head(3).to_dict()}")
            except:
                pass
    print("-" * 80)

✅ DS3 SEXUAL FORENSE: 18,661 filas | 21 columnas

📊 ANIO_HECHO                     | Tipo: Int32        | Únicos: 7     | Vacíos: 0
   • 2018                                               → 3,130
   • 2019                                               → 3,032
   • 2024                                               → 2,928
   • 2023                                               → 2,788
   • 2022                                               → 2,602
   • 2021                                               → 2,268
   • 2020                                               → 1,913
--------------------------------------------------------------------------------
📊 GENERO_VICTIMA                 | Tipo: str          | Únicos: 2     | Vacíos: 0
   • FEMENINO                                           → 16,467
   • MASCULINO                                          → 2,194
--------------------------------------------------------------------------------
📊 GRUPO_EDAD                     | Tipo: str   

In [24]:
import pandas as pd, json, warnings
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()).lower() else Path.cwd()

# ✅ CAMBIO: Fusión base + mapping para recuperar 'paths'
base_path = PROJECT_ROOT / "config" / "base_config.json"
base_cfg = {}
if base_path.exists():
    with open(base_path, 'r', encoding='utf-8') as f: base_cfg = json.load(f)
with open(PROJECT_ROOT / "config" / "mapping_config.json", 'r', encoding='utf-8') as f:
    spec_cfg = json.load(f)
cfg = {**base_cfg, **spec_cfg}  # paths se inyecta desde base

CLEAN_DIR = (PROJECT_ROOT / cfg["paths"]["cleaned_dir"]).resolve()
df = pd.read_parquet(CLEAN_DIR / "forense_limpio.parquet")

print(f"✅ DS4 VIF FORENSE: {len(df):,} filas | {len(df.columns)} columnas\n")
print("=" * 80)

for col in df.columns:
    uniq = df[col].dropna()
    print(f"📊 {col.upper():<30} | Tipo: {str(df[col].dtype):<12} | Únicos: {uniq.nunique():<5} | Vacíos: {df[col].isna().sum():,}")

    if len(uniq) > 0:
        top = uniq.value_counts().head(30)
        for val, cnt in top.items():
            print(f"   • {str(val):<50} → {cnt:,}")

        # ✅ Solo busca "sin info" en columnas de texto/categoría
        if df[col].dtype == 'object' or str(df[col].dtype) == 'string' or str(df[col].dtype) == 'category':
            try:
                # Regex con grupos sin captura (?:) para evitar warning
                sin_pattern = r"(?:sin informaci[oó]n|no aplica|no reportado|no registrado|n/a|na|null|vac[ií]o|s/i)"
                sin_info = uniq[uniq.astype(str).str.contains(sin_pattern, case=False, na=False, regex=True)]
                if 0 < len(sin_info) < len(uniq) * 0.95:
                    print(f"   ⚠️  Posibles 'sin info': {sin_info.value_counts().head(3).to_dict()}")
            except:
                pass
    print("-" * 80)

✅ DS4 VIF FORENSE: 15,418 filas | 21 columnas

📊 ANIO_HECHO                     | Tipo: Int32        | Únicos: 7     | Vacíos: 0
   • 2018                                               → 3,065
   • 2019                                               → 2,978
   • 2023                                               → 2,521
   • 2024                                               → 1,998
   • 2022                                               → 1,983
   • 2021                                               → 1,470
   • 2020                                               → 1,403
--------------------------------------------------------------------------------
📊 GENERO_VICTIMA                 | Tipo: str          | Únicos: 2     | Vacíos: 0
   • FEMENINO                                           → 9,637
   • MASCULINO                                          → 5,781
--------------------------------------------------------------------------------
📊 CICLO_VITAL                    | Tipo: str       

In [25]:
#validacion de los archivos forense y seforense en el forense analyzer
import sys
import logging
from pathlib import Path

%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

FORENSE_CONFIG = str(PROJECT_ROOT / "config" / "forense_analyzer_config.json")

from forense_analyzer import ForenseAnalyzer

print("=" * 60)
print("VERIFICACIÓN — ForenseAnalyzer Steps 1 & 2")
print("=" * 60)

# Step 1: initialization
fa = ForenseAnalyzer(config_path=FORENSE_CONFIG)

# Step 2: load and validate both datasets
df4 = fa._load_and_validate("ds4")
df3 = fa._load_and_validate("ds3")

# Assertions
assert df4.shape == (15418, 21), f"DS4 shape inesperado: {df4.shape}"
assert df3.shape == (18661, 21), f"DS3 shape inesperado: {df3.shape}"
assert set(fa.ds4_config["columns_available"]).issubset(set(df4.columns))
assert set(fa.ds3_config["columns_available"]).issubset(set(df3.columns))

print()
print(f"forense | shape: {df4.shape} | cols: {sorted(df4.columns.tolist())}")
print(f"seforense | shape: {df3.shape} | cols: {sorted(df3.columns.tolist())}")
print()
print("✅ ForenseAnalyzer Steps 1 & 2 — PASS")

2026-06-10 10:41:15,449 - INFO - ForenseAnalyzer initialized | config: /Users/anaaguirre/Documents/Cicatrices_invisibles/config/forense_analyzer_config.json
2026-06-10 10:41:15,454 - INFO - Loaded ds4: 15418 rows × 21 columns
2026-06-10 10:41:15,460 - INFO - Loaded ds3: 18661 rows × 21 columns


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
VERIFICACIÓN — ForenseAnalyzer Steps 1 & 2

forense | shape: (15418, 21) | cols: ['agresor', 'anio_hecho', 'ciclo_vital', 'cod_municipio', 'departamento', 'dia_hecho', 'dias_incapacidad', 'dimension_agresor', 'dimension_discapacidad', 'dimension_escenario', 'dimension_etnia', 'discapacidad', 'escenario', 'etnia', 'factor', 'genero_victima', 'grupo_edad', 'hora_rango', 'mes_hecho', 'municipio', 'sexo_agresor']
seforense | shape: (18661, 21) | cols: ['agresor', 'anio_hecho', 'ciclo_vital', 'circunstancia', 'cod_municipio', 'departamento', 'dia_hecho', 'dimension_agresor', 'dimension_circunstancia', 'dimension_discapacidad', 'dimension_escenario', 'dimension_etnia', 'discapacidad', 'escenario', 'etnia', 'genero_victima', 'grupo_edad', 'hora_rango', 'mes_hecho', 'municipio', 'sexo_agresor']

✅ ForenseAnalyzer Steps 1 & 2 — PASS


In [26]:
import sys, logging
from pathlib import Path
import pandas as pd

%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from forense_analyzer import ForenseAnalyzer

config_path = str(PROJECT_ROOT / "config" / "forense_analyzer_config.json")
analyzer = ForenseAnalyzer(config_path=config_path)

# ── Generate all 12 tables ──────────────────────────────────────────────────
all_tables = analyzer.analyze_all()

print("\n" + "=" * 65)
print("TABLA                              SHAPE           COLUMNAS")
print("=" * 65)
for name, df in all_tables.items():
    print(f"{name:<35} {str(df.shape):<15} {df.columns.tolist()}")

# ── Assert 12 tables returned ──────────────────────────────────────────────
assert len(all_tables) == 12, f"Expected 12 tables, got {len(all_tables)}"
print(f"\n✅ analyze_all() returned {len(all_tables)} tables — OK")

# ── Assert parquet files on disk ───────────────────────────────────────────
DS4_DIR = PROJECT_ROOT / "data" / "agregados_forense"
DS3_DIR = PROJECT_ROOT / "data" / "agregados_seforense"

ds4_files = sorted(DS4_DIR.glob("*.parquet"))
ds3_files = sorted(DS3_DIR.glob("*.parquet"))

print(f"\nParquet files in {DS4_DIR.name}/  ({len(ds4_files)} files):")
for f in ds4_files:
    df = pd.read_parquet(f)
    print(f"  {f.name:<42} {df.shape}")

print(f"\nParquet files in {DS3_DIR.name}/ ({len(ds3_files)} files):")
for f in ds3_files:
    df = pd.read_parquet(f)
    print(f"  {f.name:<42} {df.shape}")

assert len(ds4_files) == 6, f"Expected 6 DS4 parquets, got {len(ds4_files)}"
assert len(ds3_files) == 6, f"Expected 6 DS3 parquets, got {len(ds3_files)}"
print("\n✅ 6 parquet files in each output directory — OK")

# ── Sanity checks ──────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("SANITY CHECKS")
print("=" * 65)

# 1. pct_femenino in [0, 100] for all tables that have it
for name in ["ds4_municipio_resumen", "ds4_interseccional",
             "ds3_municipio_resumen", "ds3_interseccional"]:
    df = all_tables[name]
    col = "pct_femenino"
    assert df[col].between(0, 100).all(), f"{name}.{col} out of [0,100]"
    print(f"✅ {name}.pct_femenino in [0,100]  (min={df[col].min():.1f}, max={df[col].max():.1f})")

# 2. pct_total sums to ~100 per department (floating-point tolerance)
for name in ["ds4_agresor", "ds4_escenario", "ds4_factor",
             "ds3_agresor", "ds3_escenario", "ds3_circunstancia"]:
    df = all_tables[name]
    dept_sums = df.groupby("departamento", observed=True)["pct_total"].sum()
    assert (dept_sums - 100).abs().lt(0.01).all(), \
        f"{name}: pct_total dept sums not ~100\n{dept_sums}"
    print(f"✅ {name}.pct_total sums to 100 per dept")

# 3. municipio_resumen: 1 row per municipality
for name in ["ds4_municipio_resumen", "ds3_municipio_resumen"]:
    df = all_tables[name]
    assert df["cod_municipio"].nunique() == len(df), \
        f"{name}: duplicate cod_municipio rows"
    print(f"✅ {name}: unique municipalities={len(df)}")

# 4. Mode columns are non-null strings
mode_checks = {
    "ds4_municipio_resumen": ["agresor_frecuente", "escenario_frecuente", "factor_mas_frecuente"],
    "ds3_municipio_resumen": ["agresor_frecuente", "circunstancia_frecuente"],
}
for name, cols in mode_checks.items():
    df = all_tables[name]
    for col in cols:
        null_count = df[col].isna().sum()
        assert null_count == 0, f"{name}.{col}: {null_count} nulls"
    print(f"✅ {name}: mode columns non-null ({cols})")

# 5. n_casos > 0 everywhere
for name, df in all_tables.items():
    if "n_casos" in df.columns:
        assert (df["n_casos"] > 0).all(), f"{name}: zero n_casos rows found"
print("✅ n_casos > 0 in all tables that have it")

# 6. Temporalidad: correct columns, no nulls in key fields
for name in ["ds4_temporalidad", "ds3_temporalidad"]:
    df = all_tables[name]
    assert set(df.columns) == {"mes_hecho", "dia_hecho", "hora_rango", "n_casos"}, \
        f"{name}: unexpected columns {df.columns.tolist()}"
    assert df[["mes_hecho", "dia_hecho", "hora_rango"]].notna().all().all(), \
        f"{name}: nulls in groupby keys"
    print(f"✅ {name}: schema OK, no nulls in groupby keys ({len(df)} combos)")

print("\n" + "=" * 65)
print("✅ ForenseAnalyzer Step 3 — analyze_all() PASS")
print("=" * 65)


2026-06-10 10:41:23,490 - INFO - ForenseAnalyzer initialized | config: /Users/anaaguirre/Documents/Cicatrices_invisibles/config/forense_analyzer_config.json
2026-06-10 10:41:23,495 - INFO - Loaded ds4: 15418 rows × 21 columns
2026-06-10 10:41:23,634 - INFO - Saved ds4_municipio_resumen: 161 rows → /Users/anaaguirre/Documents/Cicatrices_invisibles/data/agregados_forense/ds4_municipio_resumen.parquet
2026-06-10 10:41:23,638 - INFO - Saved ds4_agresor: 108 rows → /Users/anaaguirre/Documents/Cicatrices_invisibles/data/agregados_forense/ds4_agresor.parquet
2026-06-10 10:41:23,640 - INFO - Saved ds4_escenario: 34 rows → /Users/anaaguirre/Documents/Cicatrices_invisibles/data/agregados_forense/ds4_escenario.parquet
2026-06-10 10:41:23,642 - INFO - Saved ds4_factor: 34 rows → /Users/anaaguirre/Documents/Cicatrices_invisibles/data/agregados_forense/ds4_factor.parquet
2026-06-10 10:41:23,645 - INFO - Saved ds4_temporalidad: 752 rows → /Users/anaaguirre/Documents/Cicatrices_invisibles/data/agregad

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


2026-06-10 10:41:23,814 - INFO - Saved ds3_municipio_resumen: 175 rows → /Users/anaaguirre/Documents/Cicatrices_invisibles/data/agregados_seforense/ds3_municipio_resumen.parquet
2026-06-10 10:41:23,817 - INFO - Saved ds3_agresor: 175 rows → /Users/anaaguirre/Documents/Cicatrices_invisibles/data/agregados_seforense/ds3_agresor.parquet
2026-06-10 10:41:23,819 - INFO - Saved ds3_escenario: 36 rows → /Users/anaaguirre/Documents/Cicatrices_invisibles/data/agregados_seforense/ds3_escenario.parquet
2026-06-10 10:41:23,821 - INFO - Saved ds3_circunstancia: 32 rows → /Users/anaaguirre/Documents/Cicatrices_invisibles/data/agregados_seforense/ds3_circunstancia.parquet
2026-06-10 10:41:23,824 - INFO - Saved ds3_temporalidad: 755 rows → /Users/anaaguirre/Documents/Cicatrices_invisibles/data/agregados_seforense/ds3_temporalidad.parquet
2026-06-10 10:41:23,830 - INFO - Saved ds3_interseccional: 49 rows → /Users/anaaguirre/Documents/Cicatrices_invisibles/data/agregados_seforense/ds3_interseccional.par


TABLA                              SHAPE           COLUMNAS
ds4_municipio_resumen               (161, 14)       ['cod_municipio', 'municipio', 'departamento', 'total_casos', 'pct_femenino', 'pct_nna_femenino', 'pct_adultas_femenino', 'agresor_frecuente_nna_f', 'agresor_frecuente_adultas_f', 'agresor_frecuente', 'dimension_agresor_frecuente', 'escenario_frecuente', 'dimension_escenario_frecuente', 'factor_mas_frecuente']
ds4_agresor                         (108, 5)        ['departamento', 'dimension_agresor', 'ciclo_vital', 'n_casos', 'pct_total']
ds4_escenario                       (34, 4)         ['departamento', 'dimension_escenario', 'n_casos', 'pct_total']
ds4_factor                          (34, 4)         ['departamento', 'factor', 'n_casos', 'pct_total']
ds4_temporalidad                    (752, 4)        ['mes_hecho', 'dia_hecho', 'hora_rango', 'n_casos']
ds4_interseccional                  (40, 5)         ['ciclo_vital', 'dimension_etnia', 'dimension_discapacidad', 'n_casos',

In [27]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

DS4_RESUMEN = PROJECT_ROOT / "data" / "agregados_forense"  / "ds4_municipio_resumen.parquet"
DS3_RESUMEN = PROJECT_ROOT / "data" / "agregados_seforense" / "ds3_municipio_resumen.parquet"

ds4 = pd.read_parquet(DS4_RESUMEN)
ds3 = pd.read_parquet(DS3_RESUMEN)

SEP = "=" * 80

for label, df in [("DS4 — VIF FORENSE (ds4_municipio_resumen)", ds4),
                  ("DS3 — DELITO SEXUAL FORENSE (ds3_municipio_resumen)", ds3)]:
    print(SEP)
    print(label)
    print(SEP)
    print(f"Shape : {df.shape}  ({len(df)} municipios)")
    print(f"Cols  : {df.columns.tolist()}")
    print()

    # Numeric summary — includes new segmented pct columns
    pct_cols = [c for c in df.columns if c.startswith("pct_")]
    num_cols = ["total_casos"] + pct_cols
    print(df[num_cols].describe().round(2).to_string())
    print()

    # Sanity: pct_nna_femenino + pct_adultas_femenino == pct_femenino
    diff = (df["pct_nna_femenino"] + df["pct_adultas_femenino"] - df["pct_femenino"]).abs()
    print(f"pct_nna_f + pct_adultas_f == pct_femenino  (max diff: {diff.max():.4f})")
    print()

    # Breakdown by department
    dept_summary = (
        df.groupby("departamento", observed=True)
          .agg(municipios=("cod_municipio", "count"),
               total_casos=("total_casos", "sum"),
               pct_femenino_prom=("pct_femenino", "mean"),
               pct_nna_femenino_prom=("pct_nna_femenino", "mean"),
               pct_adultas_femenino_prom=("pct_adultas_femenino", "mean"))
          .round(1)
    )
    print("Por departamento:")
    print(dept_summary.to_string())
    print()

    # Top 10 rows sorted by total_casos
    print("Top 10 municipios por total_casos:")
    pct_display = [c for c in df.columns if c.startswith("pct_")]
    mode_display = [c for c in df.columns if "frecuente" in c]
    display_cols = ["municipio", "departamento", "total_casos"] + pct_display + mode_display
    print(df[display_cols].sort_values("total_casos", ascending=False).head(10).to_string(index=False))
    print()

    # Narrative for largest municipality
    top_mun = df.sort_values("total_casos", ascending=False).iloc[0]
    print(f"--- {top_mun['municipio']} ({top_mun['departamento']}) ---")
    print(f"  Total casos         : {top_mun['total_casos']}")
    print(f"  % Femenino total    : {top_mun['pct_femenino']:.1f}%")
    print(f"  % NNA femenino      : {top_mun['pct_nna_femenino']:.1f}%")
    print(f"  % Adultas femenino  : {top_mun['pct_adultas_femenino']:.1f}%")
    for col in mode_display:
        print(f"  {col:<35}: {top_mun[col]}")
    print()


DS4 — VIF FORENSE (ds4_municipio_resumen)
Shape : (161, 14)  (161 municipios)
Cols  : ['cod_municipio', 'municipio', 'departamento', 'total_casos', 'pct_femenino', 'pct_nna_femenino', 'pct_adultas_femenino', 'agresor_frecuente_nna_f', 'agresor_frecuente_adultas_f', 'agresor_frecuente', 'dimension_agresor_frecuente', 'escenario_frecuente', 'dimension_escenario_frecuente', 'factor_mas_frecuente']

       total_casos  pct_femenino  pct_nna_femenino  pct_adultas_femenino
count       161.00        161.00            161.00                161.00
mean         95.76         59.57             14.40                 45.17
std         464.02         24.55             20.48                 26.60
min           1.00          0.00              0.00                  0.00
25%           3.00         50.00              0.00                 31.58
50%           9.00         61.15              9.06                 50.00
75%          26.00         69.23             18.13                 60.00
max        5353.0

In [28]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

DS4_AGRESOR = PROJECT_ROOT / "data" / "agregados_forense"  / "ds4_agresor.parquet"
DS3_AGRESOR = PROJECT_ROOT / "data" / "agregados_seforense" / "ds3_agresor.parquet"

ds4 = pd.read_parquet(DS4_AGRESOR)
ds3 = pd.read_parquet(DS3_AGRESOR)

SEP = "=" * 80

for label, df, tag in [
    ("DS4 — VIF FORENSE (ds4_agresor)",           ds4, "ds4"),
    ("DS3 — DELITO SEXUAL FORENSE (ds3_agresor)", ds3, "ds3"),
]:
    print(SEP)
    print(label)
    print(SEP)
    print(f"Shape : {df.shape}  |  Cols: {df.columns.tolist()}")
    print()

    # Agresores únicos
    agresores = sorted(df["dimension_agresor"].unique())
    print(f"Agresores únicos ({len(agresores)}):")
    for a in agresores:
        print(f"  {a}")
    print()

    # Ciclos vitales únicos
    ciclos = sorted(df["ciclo_vital"].unique())
    print(f"Ciclos vitales ({len(ciclos)}): {ciclos}")
    print()

    # Totales por departamento (sanity: deben sumar ~100)
    dept_totals = df.groupby("departamento", observed=True)["pct_total"].sum().round(2)
    print("pct_total por departamento (debe sumar ~100):")
    print(dept_totals.to_string())
    print()

    # Top 10 combinaciones agresor × ciclo_vital × departamento por n_casos
    print("Top 10 combinaciones por n_casos:")
    top10 = (
        df.sort_values("n_casos", ascending=False)
          .head(10)
          [["departamento", "dimension_agresor", "ciclo_vital", "n_casos", "pct_total"]]
          .reset_index(drop=True)
    )
    print(top10.to_string(index=False))
    print()

    # Pivot: agresor más frecuente por departamento (mayor n_casos acumulado)
    top_agresor = (
        df.groupby(["departamento", "dimension_agresor"], observed=True)["n_casos"]
          .sum()
          .reset_index()
          .sort_values(["departamento", "n_casos"], ascending=[True, False])
          .groupby("departamento", observed=True)
          .first()
          .rename(columns={"dimension_agresor": "agresor_top", "n_casos": "casos_agresor_top"})
    )
    print("Agresor más frecuente por departamento:")
    print(top_agresor.to_string())
    print()


DS4 — VIF FORENSE (ds4_agresor)
Shape : (108, 5)  |  Cols: ['departamento', 'dimension_agresor', 'ciclo_vital', 'n_casos', 'pct_total']

Agresores únicos (6):
  FAMILIA_ENSAMBLADA
  FAMILIA_EXTENDIDA
  FAMILIA_NUCLEAR
  FAMILIA_POLITICA
  FIGURAS_CUSTODIA
  NO_REGISTRADO

Ciclos vitales (6): ['00 A 05 PRIMERA INFANCIA', '06 A 11 INFANCIA', '12 A 17 ADOLESCENCIA', '18 A 28 JUVENTUD', '29 A 59 ADULTEZ', 'MAS DE 60 ADULTO MAYOR']

pct_total por departamento (debe sumar ~100):
departamento
CAUCA              100.0
CHOCO              100.0
NARIÑO             100.0
VALLE DEL CAUCA    100.0

Top 10 combinaciones por n_casos:
   departamento dimension_agresor            ciclo_vital  n_casos  pct_total
VALLE DEL CAUCA   FAMILIA_NUCLEAR        29 A 59 ADULTEZ     2047  21.216833
VALLE DEL CAUCA FAMILIA_EXTENDIDA        29 A 59 ADULTEZ     1172  12.147595
VALLE DEL CAUCA   FAMILIA_NUCLEAR MAS DE 60 ADULTO MAYOR      876   9.079602
VALLE DEL CAUCA  FAMILIA_POLITICA        29 A 59 ADULTEZ      813 

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

master_path = PROJECT_ROOT / "data" / "master" / "master_table.parquet"
df_master = pd.read_parquet(master_path)

print(f"Shape: {df_master.shape}")
print(f"Columns: {df_master.columns.tolist()}")
print()
df_master.head(10)


Shape: (1432, 31)
Columns: ['cod_municipio', 'anio_hecho', 'municipio', 'departamento', 'pob_f_0_17', 'pob_f_18_mas', 'pob_h_0_17', 'pob_h_18_mas', 'total_f', 'total_h', 'casos_vif_nna_f', 'casos_vif_adultas_f', 'casos_sexual_nna_f', 'casos_sexual_adultas_f', 'casos_vif_nna_m', 'casos_vif_adultos_m', 'casos_sexual_nna_m', 'casos_sexual_adultos_m', 'tasa_vif_nna_f', 'tasa_vif_adultas_f', 'tasa_sexual_nna_f', 'tasa_sexual_adultas_f', 'tasa_vif_nna_m', 'tasa_vif_adultos_m', 'tasa_sexual_nna_m', 'tasa_sexual_adultos_m', 'brecha_vif_nna', 'brecha_vif_adultas', 'brecha_sexual_nna', 'brecha_sexual_adultas', 'icv_gen_f_score']



,cod_municipio,anio_hecho,municipio,departamento,pob_f_0_17,pob_f_18_mas,pob_h_0_17,pob_h_18_mas,total_f,total_h,...,tasa_sexual_adultas_f,tasa_vif_nna_m,tasa_vif_adultos_m,tasa_sexual_nna_m,tasa_sexual_adultos_m,brecha_vif_nna,brecha_vif_adultas,brecha_sexual_nna,brecha_sexual_adultas,icv_gen_f_score
0,19001,2018,POPAYAN,CAUCA,39448,125069,42195,111343,164517,153538,...,171.905108,63.988624,304.464582,49.76893,32.332522,1.861959,3.10144,6.010335,5.316786,56.363469
1,19001,2019,POPAYAN,CAUCA,39552,128200,42396,114446,167752,156842,...,163.026521,106.142089,325.9179,33.021983,13.106618,2.334372,2.917477,7.197079,12.438489,65.542587
2,19001,2020,POPAYAN,CAUCA,39683,131391,42576,117654,171074,160230,...,124.057203,79.857197,235.436109,35.231116,16.149047,2.272029,2.990219,5.22146,7.682014,48.854116
3,19001,2021,POPAYAN,CAUCA,39661,134040,42557,120180,173701,162737,...,109.668756,58.744742,257.114329,32.897056,19.970045,1.888513,2.750728,5.28845,5.491663,41.11475
4,19001,2022,POPAYAN,CAUCA,39531,136375,42441,122366,175906,164807,...,103.391384,84.823638,244.34892,21.205909,14.709968,1.699887,2.745844,7.157421,7.028661,42.074537
5,19001,2023,POPAYAN,CAUCA,39290,138500,42249,124320,177790,166569,...,98.916968,66.273758,231.660232,21.302279,17.696268,1.612968,3.244513,6.690829,5.589708,39.875937
6,19001,2024,POPAYAN,CAUCA,39056,140818,42059,126511,179874,168570,...,97.998835,38.041798,148.603679,26.153736,19.761127,3.163363,3.875548,6.16764,4.959172,37.67178
7,19001,2025,POPAYAN,CAUCA,38763,142985,41812,128590,181748,170402,...,111.89985,35.874868,162.532079,21.524921,20.219302,3.020241,3.580088,5.75284,5.534308,35.286807
8,19022,2018,ALMAGUER,CAUCA,2908,6318,2950,6272,9226,9222,...,15.827794,0.0,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,8.788095
9,19022,2019,ALMAGUER,CAUCA,2850,6367,2903,6340,9217,9243,...,47.117952,0.0,31.545741,0.0,0.0,<NA>,3.485158,<NA>,<NA>,12.983743


In [2]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent

clustering_path = PROJECT_ROOT / "data" / "master" / "tabla_clustering.parquet"
df_clustering = pd.read_parquet(clustering_path)

print(f"Shape: {df_clustering.shape}")
print(f"Columns: {df_clustering.columns.tolist()}")
print()
df_clustering.head(10)


Shape: (179, 24)
Columns: ['cod_municipio', 'casos_vif_nna_f', 'casos_vif_adultas_f', 'casos_sexual_nna_f', 'casos_sexual_adultas_f', 'casos_vif_nna_m', 'casos_vif_adultos_m', 'casos_sexual_nna_m', 'casos_sexual_adultos_m', 'tasa_vif_nna_f', 'tasa_vif_adultas_f', 'tasa_sexual_nna_f', 'tasa_sexual_adultas_f', 'tasa_vif_nna_m', 'tasa_vif_adultos_m', 'tasa_sexual_nna_m', 'tasa_sexual_adultos_m', 'brecha_vif_nna', 'brecha_vif_adultas', 'brecha_sexual_nna', 'brecha_sexual_adultas', 'icv_gen_f_score', 'municipio', 'departamento']



,cod_municipio,casos_vif_nna_f,casos_vif_adultas_f,casos_sexual_nna_f,casos_sexual_adultas_f,casos_vif_nna_m,casos_vif_adultos_m,casos_sexual_nna_m,casos_sexual_adultos_m,tasa_vif_nna_f,...,tasa_vif_adultos_m,tasa_sexual_nna_m,tasa_sexual_adultos_m,brecha_vif_nna,brecha_vif_adultas,brecha_sexual_nna,brecha_sexual_adultas,icv_gen_f_score,municipio,departamento
0,19001,56.125,984.0,72.625,163.75,28.25,285.25,12.75,23.125,142.384528,...,238.759729,30.138241,19.243112,2.231667,3.150732,6.185757,6.7551,45.847998,POPAYAN,CAUCA
1,19022,1.125,8.75,0.875,2.625,0.875,1.125,0.0,0.0,40.087481,...,17.498096,0.0,0.0,2.211203,5.232785,<NA>,<NA>,10.28347,ALMAGUER,CAUCA
2,19050,0.25,10.25,5.875,8.0,0.0,0.875,0.625,0.625,5.579851,...,9.486705,13.286627,6.742252,<NA>,9.89257,7.039833,5.204532,13.670244,ARGELIA,CAUCA
3,19075,0.125,12.0,4.0,3.75,0.0,1.875,0.0,0.625,3.614806,...,25.441443,0.0,8.462721,<NA>,7.569141,<NA>,3.965578,12.004521,BALBOA,CAUCA
4,19100,0.75,18.5,5.0,6.875,0.0,3.75,0.25,1.125,15.313575,...,26.726604,4.419535,8.020176,<NA>,7.007367,5.513591,4.581905,11.785956,BOLIVAR,CAUCA
5,19110,0.5,12.75,2.375,6.125,0.25,2.25,0.5,0.5,9.419321,...,21.354878,8.901577,4.812183,1.053947,6.911881,2.105522,6.33444,8.074246,BUENOS AIRES,CAUCA
6,19130,2.5,40.0,7.75,18.375,0.75,6.0,0.5,1.625,33.773124,...,41.173734,6.267835,11.016876,2.6786,7.439146,7.501467,13.800859,20.327213,CAJIBIO,CAUCA
7,19137,1.125,22.75,6.375,8.625,0.75,5.0,0.75,0.5,14.25764,...,36.367117,9.182846,3.606323,0.129921,7.917615,6.463318,7.298596,12.270803,CALDONO,CAUCA
8,19142,2.0,44.125,3.75,6.75,1.375,12.0,0.25,0.625,42.72913,...,117.103944,5.001333,5.993375,1.058744,3.495272,5.878413,4.157144,20.722524,CALOTO,CAUCA
9,19212,1.0,19.125,3.875,4.375,0.625,4.5,0.5,0.875,26.381119,...,47.640732,12.261211,9.280165,0.404879,8.925357,3.783384,1.013446,14.666002,CORINTO,CAUCA


In [3]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
MASTER = PROJECT_ROOT / "data" / "master"

files = {
    "tabla_clustering_final.parquet":       MASTER / "tabla_clustering_final.parquet",
    "tabla_clustering_con_clusters.parquet": MASTER / "tabla_clustering_con_clusters.parquet",
}

SEP = "=" * 80

for name, path in files.items():
    df = pd.read_parquet(path)
    print(SEP)
    print(f"  {name}")
    print(SEP)
    print(f"Shape   : {df.shape}")
    print(f"Columns : {df.columns.tolist()}")
    print()
    display(df.head(10))
    print()


  tabla_clustering_final.parquet
Shape   : (177, 26)
Columns : ['cod_municipio', 'casos_vif_nna_f', 'casos_vif_adultas_f', 'casos_sexual_nna_f', 'casos_sexual_adultas_f', 'casos_vif_nna_m', 'casos_vif_adultos_m', 'casos_sexual_nna_m', 'casos_sexual_adultos_m', 'tasa_vif_nna_f', 'tasa_vif_adultas_f', 'tasa_sexual_nna_f', 'tasa_sexual_adultas_f', 'tasa_vif_nna_m', 'tasa_vif_adultos_m', 'tasa_sexual_nna_m', 'tasa_sexual_adultos_m', 'brecha_vif_nna', 'brecha_vif_adultas', 'brecha_sexual_nna', 'brecha_sexual_adultas', 'icv_gen_f_score', 'municipio', 'departamento', 'cluster_id', 'cluster_name']



,cod_municipio,casos_vif_nna_f,casos_vif_adultas_f,casos_sexual_nna_f,casos_sexual_adultas_f,casos_vif_nna_m,casos_vif_adultos_m,casos_sexual_nna_m,casos_sexual_adultos_m,tasa_vif_nna_f,...,tasa_sexual_adultos_m,brecha_vif_nna,brecha_vif_adultas,brecha_sexual_nna,brecha_sexual_adultas,icv_gen_f_score,municipio,departamento,cluster_id,cluster_name
0,19001,56.125,984.0,72.625,163.75,28.25,285.25,12.75,23.125,142.384528,...,19.243112,2.231667,3.150732,6.185757,6.7551,45.847998,POPAYAN,CAUCA,1,🔴 Alta severidad
1,19022,1.125,8.75,0.875,2.625,0.875,1.125,0.0,0.0,40.087481,...,0.0,2.211203,5.232785,<NA>,<NA>,10.28347,ALMAGUER,CAUCA,0,🟠 Moderada/Baja
2,19050,0.25,10.25,5.875,8.0,0.0,0.875,0.625,0.625,5.579851,...,6.742252,<NA>,9.89257,7.039833,5.204532,13.670244,ARGELIA,CAUCA,1,🔴 Alta severidad
3,19075,0.125,12.0,4.0,3.75,0.0,1.875,0.0,0.625,3.614806,...,8.462721,<NA>,7.569141,<NA>,3.965578,12.004521,BALBOA,CAUCA,1,🔴 Alta severidad
4,19100,0.75,18.5,5.0,6.875,0.0,3.75,0.25,1.125,15.313575,...,8.020176,<NA>,7.007367,5.513591,4.581905,11.785956,BOLIVAR,CAUCA,1,🔴 Alta severidad
5,19110,0.5,12.75,2.375,6.125,0.25,2.25,0.5,0.5,9.419321,...,4.812183,1.053947,6.911881,2.105522,6.33444,8.074246,BUENOS AIRES,CAUCA,0,🟠 Moderada/Baja
6,19130,2.5,40.0,7.75,18.375,0.75,6.0,0.5,1.625,33.773124,...,11.016876,2.6786,7.439146,7.501467,13.800859,20.327213,CAJIBIO,CAUCA,1,🔴 Alta severidad
7,19137,1.125,22.75,6.375,8.625,0.75,5.0,0.75,0.5,14.25764,...,3.606323,0.129921,7.917615,6.463318,7.298596,12.270803,CALDONO,CAUCA,1,🔴 Alta severidad
8,19142,2.0,44.125,3.75,6.75,1.375,12.0,0.25,0.625,42.72913,...,5.993375,1.058744,3.495272,5.878413,4.157144,20.722524,CALOTO,CAUCA,1,🔴 Alta severidad
9,19212,1.0,19.125,3.875,4.375,0.625,4.5,0.5,0.875,26.381119,...,9.280165,0.404879,8.925357,3.783384,1.013446,14.666002,CORINTO,CAUCA,1,🔴 Alta severidad



  tabla_clustering_con_clusters.parquet
Shape   : (179, 18)
Columns : ['cod_municipio', 'casos_vif_nna_f', 'casos_vif_adultas_f', 'casos_sexual_nna_f', 'casos_sexual_adultas_f', 'casos_vif_nna_m', 'casos_vif_adultos_m', 'casos_sexual_nna_m', 'casos_sexual_adultos_m', 'tasa_vif_nna_f', 'tasa_vif_adultas_f', 'tasa_sexual_nna_f', 'tasa_sexual_adultas_f', 'icv_gen_f_score', 'municipio', 'departamento', 'cluster_id', 'cluster_name']



,cod_municipio,casos_vif_nna_f,casos_vif_adultas_f,casos_sexual_nna_f,casos_sexual_adultas_f,casos_vif_nna_m,casos_vif_adultos_m,casos_sexual_nna_m,casos_sexual_adultos_m,tasa_vif_nna_f,tasa_vif_adultas_f,tasa_sexual_nna_f,tasa_sexual_adultas_f,icv_gen_f_score,municipio,departamento,cluster_id,cluster_name
0,19001,56.125,984.0,72.625,163.75,28.25,285.25,12.75,23.125,377.961289,2268.715184,490.682071,378.351066,7.144355,POPAYAN,CAUCA,0,🔴 Alta severidad (Crisis estructural)
1,19022,1.125,8.75,0.875,2.625,0.875,1.125,0.0,0.0,214.01651,726.183996,172.734983,221.618183,3.121993,ALMAGUER,CAUCA,0,🔴 Alta severidad (Crisis estructural)
2,19050,0.25,10.25,5.875,8.0,0.0,0.875,0.625,0.625,29.629534,429.517048,699.18193,335.583913,4.35205,BALBOA,CAUCA,0,🔴 Alta severidad (Crisis estructural)
3,19075,0.125,12.0,4.0,3.75,0.0,1.875,0.0,0.625,7.88258,317.367837,257.285487,99.70945,1.663112,BOLIVAR,CAUCA,1,🟠 Severidad moderada (Riesgo balanceado)
4,19100,0.75,18.5,5.0,6.875,0.0,3.75,0.25,1.125,60.349001,434.393882,402.288543,161.777352,2.778369,BUENOS AIRES,CAUCA,0,🔴 Alta severidad (Crisis estructural)
5,19110,0.5,12.75,2.375,6.125,0.25,2.25,0.5,0.5,113.954343,1172.148566,570.639447,578.71426,5.848774,NaN,CAUCA,0,🔴 Alta severidad (Crisis estructural)
6,19130,2.5,40.0,7.75,18.375,0.75,6.0,0.5,1.625,348.798595,1870.982806,1083.297407,858.290348,10.511744,CAJIBIO,CAUCA,0,🔴 Alta severidad (Crisis estructural)
7,19137,1.125,22.75,6.375,8.625,0.75,5.0,0.75,0.5,135.617536,1420.348891,762.136046,531.774733,6.745867,NaN,CAUCA,0,🔴 Alta severidad (Crisis estructural)
8,19142,2.0,44.125,3.75,6.75,1.375,12.0,0.25,0.625,125.368277,1021.613873,238.402522,155.781555,3.026745,CALDONO,CAUCA,0,🔴 Alta severidad (Crisis estructural)
9,19212,1.0,19.125,3.875,4.375,0.625,4.5,0.5,0.875,43.976126,350.684258,170.556592,81.221794,1.47204,CORINTO,CAUCA,1,🟠 Severidad moderada (Riesgo balanceado)


In [5]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
df_sexuales = pd.read_parquet(PROJECT_ROOT / "data" / "cleaned" / "sexuales_limpio.parquet")

print(f"Shape: {df_sexuales.shape}")
print(f"Columns: {df_sexuales.columns.tolist()}")
display(df_sexuales.head(10))


Shape: (34592, 11)
Columns: ['departamento', 'municipio', 'cod_municipio', 'fecha_hecho', 'anio_hecho', 'mes_hecho', 'dia_hecho', 'genero_victima', 'grupo_etario', 'cantidad', 'dimension_delito']


,departamento,municipio,cod_municipio,fecha_hecho,anio_hecho,mes_hecho,dia_hecho,genero_victima,grupo_etario,cantidad,dimension_delito
0,VALLE DEL CAUCA,YOTOCO,76890,2019-03-18,2019,MARZO,LUNES,FEMENINO,ADULTOS,1,VIOLENCIA_SEXUAL_DIRECTA
1,VALLE DEL CAUCA,PALMIRA,76520,2020-09-18,2020,SEPTIEMBRE,VIERNES,FEMENINO,ADULTOS,1,ABUSO_SEXUAL_MENOR_14
2,NARIÑO,ILES,52352,2022-01-24,2022,ENERO,LUNES,FEMENINO,ADULTOS,1,VIOLENCIA_SEXUAL_DIRECTA
3,VALLE DEL CAUCA,CALI,76001,2024-03-12,2024,MARZO,MARTES,FEMENINO,ADULTOS,1,VIOLENCIA_SEXUAL_DIRECTA
4,VALLE DEL CAUCA,CALI,76001,2024-02-01,2024,FEBRERO,JUEVES,FEMENINO,ADULTOS,1,VIOLENCIA_SEXUAL_DIRECTA
5,CAUCA,SOTARA,19760,2024-02-20,2024,FEBRERO,MARTES,FEMENINO,ADULTOS,1,VIOLENCIA_SEXUAL_DIRECTA
6,CAUCA,SANTANDER DE QUILICHAO,19698,2024-03-18,2024,MARZO,LUNES,FEMENINO,ADULTOS,1,VIOLENCIA_SEXUAL_DIRECTA
7,VALLE DEL CAUCA,CANDELARIA,76130,2024-03-06,2024,MARZO,MIERCOLES,FEMENINO,ADULTOS,1,VIOLENCIA_SEXUAL_DIRECTA
8,VALLE DEL CAUCA,CALI,76001,2024-03-15,2024,MARZO,VIERNES,MASCULINO,ADULTOS,1,VIOLENCIA_SEXUAL_DIRECTA
9,CAUCA,ARGELIA,19050,2024-02-20,2024,FEBRERO,MARTES,MASCULINO,ADOLESCENTES,1,VIOLENCIA_SEXUAL_DIRECTA


In [7]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "config").exists() else Path.cwd().parent
df_sexuales = pd.read_parquet(PROJECT_ROOT / "data" / "cleaned" / "sexuales_limpio.parquet")

df_fem = df_sexuales[df_sexuales["genero_victima"].str.upper() == "FEMENINO"]

dist = (
    df_fem["dimension_delito"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename_axis("dimension_delito")
    .reset_index(name="porcentaje")
)

print(f"Víctimas femeninas: {len(df_fem):,} de {len(df_sexuales):,} total")
print(f"Categorías en dimension_delito: {dist.shape[0]}")
display(dist)


Víctimas femeninas: 29,595 de 34,592 total
Categorías en dimension_delito: 8


,dimension_delito,porcentaje
0,ABUSO_SEXUAL_MENOR_14,50.61
1,VIOLENCIA_SEXUAL_DIRECTA,31.36
2,ACOSO_SEXUAL,8.80
3,ABUSO_INCAPAZ_RESISTIR,5.07
4,EXPLOTACION_SEXUAL_DIGITAL_MENORES,3.16
5,EXPLOTACION_SEXUAL_COMERCIAL_MENOR,0.52
6,EXPLOTACION_SEXUAL_COMERCIAL,0.48
7,OTROS,0.00
